In [490]:
# from google.colab import drive
# drive.mount('/content/drive')

In [491]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use(style='ggplot')

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import MinMaxScaler, StandardScaler

from sklearn.preprocessing import OneHotEncoder, LabelEncoder

from sklearn.preprocessing import PolynomialFeatures

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# change float format in data
pd.options.display.float_format = '{:,.2f}'.format

In [492]:
# Change Path if google colab is used
import os
import sys

if 'google.colab' in sys.modules:
    path = '/content/drive/MyDrive/Grokking Machine Learning/03- Python Machine Learning/W15/Home Loan Approval/loan_sanction_train.csv'
else:
    path = 'loan_sanction_train.csv'

In [493]:
data = pd.read_csv(path)
data.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.00,NaN,360.00,1.00,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,"1,508.00",128.00,360.00,1.00,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.00,66.00,360.00,1.00,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,"2,358.00",120.00,360.00,1.00,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.00,141.00,360.00,1.00,Urban,Y


In [494]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [495]:
data.describe(include = 'number')

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.00,614.00,592.00,600.00,564.00
mean,"5,403.46","1,621.25",146.41,342.00,0.84
std,"6,109.04","2,926.25",85.59,65.12,0.36
min,150.00,0.00,9.00,12.00,0.00
25%,"2,877.50",0.00,100.00,360.00,1.00
50%,"3,812.50","1,188.50",128.00,360.00,1.00
75%,"5,795.00","2,297.25",168.00,360.00,1.00
max,"81,000.00","41,667.00",700.00,480.00,1.00


In [496]:
data.describe(include = 'object')

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,Property_Area,Loan_Status
count,614,601,611,599,614,582,614,614
unique,614,2,2,4,2,2,3,2
top,LP001002,Male,Yes,0,Graduate,No,Semiurban,Y
freq,1,489,398,345,480,500,233,422


In [497]:
# Pandas Profiling for quick data analysis
# import pandas_profiling as pp

# # Report for data analysis
# report = pp.ProfileReport(data)
# report.to_file('report.html')

In [498]:
data.columns = [col.replace(' ', '') for col in data.columns]

In [499]:
# Check the sum of missing values for each column
data.isna().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [500]:
# Removing missing values
data = data.dropna(axis=0)

In [501]:
# Check missing values again
data.isnull().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [502]:
# Check Duplicate Values
data.duplicated().sum()

0

In [503]:
# Dropping the Loan_ID column
data.drop(['Loan_ID'], axis = 1, inplace = True)

In [504]:
data['Dependents'].unique()

array(['1', '0', '2', '3+'], dtype=object)

In [505]:
data['Dependents'] = data['Dependents'].map({'0':0, '1':1, '2':2, '3+':3})

In [506]:
data['Dependents'].unique()

array([1, 0, 2, 3], dtype=int64)

In [507]:
# Creating a list including the names of numerical data
num_features = data.select_dtypes(include = 'number').columns.tolist()

print(f'Numerical Features: {num_features}')

Numerical Features: ['Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']


In [508]:
cat_features = data.select_dtypes(include = 'object').columns.tolist()

print(f'Numerical Features: {cat_features}')

Numerical Features: ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']


In [509]:
# check the outliers
plt.figure(figsize=(14,7))
for i,v in enumerate(num_features):
    plt.subplot(2, 3, i+1)
    sns.boxplot(data = data, x = v, color = 'red')
plt.show()

In [510]:
# Remove outliers for Applicant Income
data.drop(data[data['ApplicantIncome'] > 30000].index, axis = 0, inplace = True)

# Remove outliers for Co-applicant Income
data.drop(data[data['CoapplicantIncome'] > 15000].index, axis = 0, inplace = True)


In [511]:
data['Loan_Status'].value_counts(normalize=True)

Y   0.69
N   0.31
Name: Loan_Status, dtype: float64

In [512]:
for cat in cat_features:
    print(cat)
    print(data[cat].unique())
    print()

Gender
['Male' 'Female']

Married
['Yes' 'No']

Education
['Graduate' 'Not Graduate']

Self_Employed
['No' 'Yes']

Property_Area
['Rural' 'Urban' 'Semiurban']

Loan_Status
['N' 'Y']



In [513]:
sns.pairplot(data,hue = 'Loan_Status')
plt.show()

In [514]:
data['Loan_Status'] = data['Loan_Status'].map({'Y':1, 'N':0})

In [515]:
data.corr()

,Dependents,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status
Dependents,1.00,0.08,0.03,0.14,-0.09,-0.00,0.04
ApplicantIncome,0.08,1.00,-0.15,0.60,0.01,0.05,-0.02
CoapplicantIncome,0.03,-0.15,1.00,0.32,-0.03,-0.06,-0.00
LoanAmount,0.14,0.60,0.32,1.00,0.08,0.00,-0.07
Loan_Amount_Term,-0.09,0.01,-0.03,0.08,1.00,0.02,-0.00
Credit_History,-0.00,0.05,-0.06,0.00,0.02,1.00,0.54
Loan_Status,0.04,-0.02,-0.00,-0.07,-0.00,0.54,1.00


In [516]:
# Splitting data to X (Features) and y (label)
X = data.drop('Loan_Status', axis = 1)
y = data['Loan_Status']

In [517]:
# Splitting data to X_train, y_train, X_test, and y_test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size  = 0.2, random_state = 42)

In [518]:
print('X training shape = ', X_train.shape)
print('Y training shape = ', y_train.shape)
print('X test shape     = ', X_test.shape)
print('Y test shape     = ', y_test.shape)

X training shape =  (377, 11)
Y training shape =  (377,)
X test shape     =  (95, 11)
Y test shape     =  (95,)


In [519]:
print(f'Numerical Features  : {num_features}')
print(f'Categorical Features: {cat_features}')

Numerical Features  : ['Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
Categorical Features: ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']


In [520]:
num_to_scale = [ 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
cat_to_encode = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']

In [521]:
# Create a transformer to encoding the categorical features using OneHotEncoder and rescaling the numerical features using StandardScaler
transformer = ColumnTransformer([('numerical', StandardScaler(), num_to_scale),
                                 ('categorical', OneHotEncoder(drop = 'first'), cat_to_encode)], remainder = 'passthrough')

# Fit the data
transformer.fit(X_train)

ColumnTransformer(remainder='passthrough',
                  transformers=[('numerical', StandardScaler(),
                                 ['ApplicantIncome', 'CoapplicantIncome',
                                  'LoanAmount', 'Loan_Amount_Term']),
                                ('categorical', OneHotEncoder(drop='first'),
                                 ['Gender', 'Married', 'Education',
                                  'Self_Employed', 'Property_Area'])])

In [522]:
# Transform the X training data
X_train = transformer.transform(X_train)

# Transform the X testing data
X_test = transformer.transform(X_test)

In [523]:
X_train.shape

(377, 12)

# Modeling

In [524]:
# Instantiating Logistic Regression Classifier
lr = LogisticRegression()

# Fit the data
lr.fit(X_train, y_train)

# Accuracy score
print('Accuracy score for Logistic Regression = ', lr.score(X_test, y_test))

Accuracy score for Logistic Regression =  0.8631578947368421


In [525]:
# Cross Validation for Logistic Regression Model
lr_scores = cross_val_score(lr,X_train, y_train, cv = 5)

print(f'Logistic Regression Accuracy          :  {lr_scores}')
print(f'Logistic Regression Accuracy Mean     :  {round(lr_scores.mean() * 100, 2)} %')
print(f'Logistic Regression Standard Deviation:  {round(lr_scores.std(), 2)}')

Logistic Regression Accuracy          :  [0.80263158 0.80263158 0.78666667 0.73333333 0.82666667]
Logistic Regression Accuracy Mean     :  79.04 %
Logistic Regression Standard Deviation:  0.03


In [526]:
# classification report
print(classification_report(y_test, lr.predict(X_test)))

              precision    recall  f1-score   support

           0       0.92      0.48      0.63        23
           1       0.86      0.99      0.92        72

    accuracy                           0.86        95
   macro avg       0.89      0.73      0.77        95
weighted avg       0.87      0.86      0.85        95



In [527]:
# Instantiating Decision Tree Classifier
tree = DecisionTreeClassifier()

# Define the parameter grid to search
param_grid = {'max_depth': np.arange(1, 10)}

# Perform grid search
tree_grid_search = GridSearchCV(estimator = tree, param_grid = param_grid, cv = 3)

# Fit the data
tree_grid_search.fit(X_train, y_train)

# Print the best parameters and score
print('Best parameters: ', tree_grid_search.best_params_)
print('Best score     : ', tree_grid_search.best_score_)

Best parameters:  {'max_depth': 1}
Best score     :  0.8010793650793652


In [528]:
# Instantiating Random Forest Classifier
rf = RandomForestClassifier()

# Define the parameter grid to search
param_grid = {'n_estimators': [50, 100, 200, 300],
              'max_depth': [1, 3, 5, 8, 11]}

# Perform grid search
rf_grid_search = GridSearchCV(estimator = rf, param_grid = param_grid, cv = 3)

# Fit the data
rf_grid_search.fit(X_train, y_train)

# Print the best parameters and score
print('Best parameters: ', rf_grid_search.best_params_)
print('Best score     : ', rf_grid_search.best_score_)

Best parameters:  {'max_depth': 5, 'n_estimators': 300}
Best score     :  0.8117248677248677


In [529]:
print('Classification Report')
print(classification_report(y_test, rf_grid_search.predict(X_test)))

Classification Report
              precision    recall  f1-score   support

           0       0.92      0.48      0.63        23
           1       0.86      0.99      0.92        72

    accuracy                           0.86        95
   macro avg       0.89      0.73      0.77        95
weighted avg       0.87      0.86      0.85        95



In [530]:
print('Confusion Matrix')
cm = confusion_matrix(y_test, rf_grid_search.predict(X_test))
sns.heatmap(cm, annot = True, cmap = 'Reds', fmt = 'd')

Confusion Matrix


<Axes: >

In [531]:
# XGBoost
from xgboost import XGBClassifier

# Instantiating XGBClassifier
xgb = XGBClassifier()

# Define the parameter grid to search
param_grid = {'n_estimators': [50, 100, 200, 300],
                'max_depth': [1, 3, 5, 8, 11],
                'learning_rate': [0.01, 0.1, 0.2, 0.3],
                'gamma': [0, 0.1, 0.2, 0.3]}
# Perform grid search
xgb_grid_search = GridSearchCV(estimator = xgb, param_grid = param_grid, cv = 3)

# Fit the data
xgb_grid_search.fit(X_train, y_train)

# Print the best parameters and score
print('Best parameters: ', xgb_grid_search.best_params_)
print('Best score     : ', xgb_grid_search.best_score_)


Best parameters:  {'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50}
Best score     :  0.8037248677248678


In [532]:
# Voting Classifier (Logistic Regression, Random Forest)
from sklearn.ensemble import VotingClassifier

# Instantiating Logistic Regression Classifier
lr = LogisticRegression()

# Instantiating Random Forest Classifier
rf = RandomForestClassifier()

# Instantiating Voting Classifier
vc = VotingClassifier(estimators = [('lr', lr), ('rf', rf)], voting = 'soft')

# Grid Search for Voting Classifier
param_grid = {'lr__C': [0.1, 1, 10, 100],
                'rf__n_estimators': [50, 100, 200, 300],
                'rf__max_depth': [1, 3, 5, 8, 11]}
# Perform grid search
vc_grid_search = GridSearchCV(estimator = vc, param_grid = param_grid, cv = 3)

# Fit the data
vc_grid_search.fit(X_train, y_train)

# Print the best parameters and score
print('Best parameters: ', vc_grid_search.best_params_)
print('Best score     : ', vc_grid_search.best_score_)

Best parameters:  {'lr__C': 0.1, 'rf__max_depth': 5, 'rf__n_estimators': 100}
Best score     :  0.8010793650793652


In [533]:
print('Classification Report')
print(classification_report(y_test, vc_grid_search.predict(X_test)))

Classification Report
              precision    recall  f1-score   support

           0       0.92      0.48      0.63        23
           1       0.86      0.99      0.92        72

    accuracy                           0.86        95
   macro avg       0.89      0.73      0.77        95
weighted avg       0.87      0.86      0.85        95



In [534]:
# ROC Curve
# Logistic Regression
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)
lr_probs = lr_probs[:, 1]

# Random Forest
rf_grid_search.fit(X_train, y_train)
rf_probs = rf_grid_search.predict_proba(X_test)
rf_probs = rf_probs[:, 1]

# XGBoost
xgb_grid_search.fit(X_train, y_train)
xgb_probs = xgb_grid_search.predict_proba(X_test)
xgb_probs = xgb_probs[:, 1]

# Voting Classifier
vc_probs = vc_grid_search.predict_proba(X_test)
vc_probs = vc_probs[:, 1]

# Calculate scores
lr_auc = roc_auc_score(y_test, lr_probs)
rf_auc = roc_auc_score(y_test, rf_probs)
xgb_auc = roc_auc_score(y_test, xgb_probs)
vc_auc = roc_auc_score(y_test, vc_probs)

# Print scores
print('Logistic Regression: ROC AUC=%.3f' % (lr_auc))
print('Random Forest      : ROC AUC=%.3f' % (rf_auc))
print('XGBoost            : ROC AUC=%.3f' % (xgb_auc))
print('Voting Classifier  : ROC AUC=%.3f' % (vc_auc))

# Calculate roc curves
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_probs)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_probs)
vc_fpr, vc_tpr, _ = roc_curve(y_test, vc_probs)

# Plot the roc curve for the model

plt.figure(figsize=(10, 7))
plt.plot(lr_fpr, lr_tpr, marker='.', label='Logistic Regression')
plt.plot(rf_fpr, rf_tpr, marker='.', label='Random Forest')
plt.plot(xgb_fpr, xgb_tpr, marker='.', label='XGBoost')
plt.plot(vc_fpr, vc_tpr, marker='.', label='Voting Classifier')

# axis labels
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

# show the legend
plt.legend()

# show the plot
plt.show()

Logistic Regression: ROC AUC=0.783
Random Forest      : ROC AUC=0.755
XGBoost            : ROC AUC=0.757
Voting Classifier  : ROC AUC=0.774


In [535]:
# Voting Classifier (Logistic Regression, SVM)
from sklearn.svm import SVC

# Instantiating Logistic Regression Classifier
lr = LogisticRegression()

# Instantiating SVM Classifier
svm = SVC(probability = True, kernel='linear')

# Instantiating Voting Classifier
vc = VotingClassifier(estimators = [('lr', lr), ('svm', svm)], voting = 'soft')

# Grid Search for Voting Classifier
param_grid = {'lr__C': [0.1, 1, 10, 100, 1000, 10000],
                'svm__C': [0.1, 1, 10, 100,1000]}

# Perform grid search
vc_grid_search = GridSearchCV(estimator = vc, param_grid = param_grid, cv = 3)

# Fit the data
vc_grid_search.fit(X_train, y_train)

# Print the best parameters and score
print('Best parameters: ', vc_grid_search.best_params_)
print('Best score     : ', vc_grid_search.best_score_)



In [ ]:
vc_grid_search.score(X_test, y_test)

0.8631578947368421

In [ ]:
print('Classification Report')
print(classification_report(y_test, vc_grid_search.predict(X_test)))

Classification Report
              precision    recall  f1-score   support

           0       0.92      0.48      0.63        23
           1       0.86      0.99      0.92        72

    accuracy                           0.86        95
   macro avg       0.89      0.73      0.77        95
weighted avg       0.87      0.86      0.85        95



In [ ]:
# Precision Recall Curve
from sklearn.metrics import precision_recall_curve

# Logistic Regression
lr.fit(X_train, y_train)

lr_probs = lr.predict_proba(X_test)
lr_probs = lr_probs[:, 1]

thresholds, lr_precision, lr_recall = precision_recall_curve(y_test, lr_probs)
df_lr = pd.DataFrame({'thresholds': thresholds, 'precision': lr_precision[:-1], 'recall': lr_recall[:-1]})
df_lr.head()

In [ ]:
# choose the threshold with the largest f score
fscore = (2 * df_lr['precision'] * df_lr['recall']) / (df_lr['precision'] + df_lr['recall'])
ix = np.argmax(fscore)
print('Best Threshold=%f, F-Score=%.3f' % (df_lr['thresholds'].iloc[ix], fscore[ix]))

In [ ]:
# choose the threshold with the largest precision
ix = np.argmax(df_lr['precision'])
print('Best Threshold=%f, Precision=%.3f' % (df_lr['thresholds'].iloc[ix], df_lr['precision'].iloc[ix]))

In [ ]:
# choose the threshold with the 90% precision
ix = np.argmin(np.abs(df_lr['precision'] - 0.90))
print('Best Threshold=%f, Recall=%.3f' % (df_lr['thresholds'].iloc[ix], df_lr['recall'].iloc[ix]))

In [ ]:
# Grid Search with dictionary
# Define the parameter grid to search on Logistic Regression
param_grid = {'C': [0.1, 1, 10, 100, 1000, 10000],
                'penalty': ['l1', 'l2'],
                'solver': ['liblinear', 'lbfgs']}

# Grid Search with List of Dictionaries
# Define the parameter grid to search on Logistic Regression
param_grid = [
                {'C': [0.1, 1, 10], 'penalty': ['l2', None], 'solver': ['newton-cg', 'lbfgs']},
                {'C': [0.1, 1, 10, 100, 1000, 10000], 'penalty': ['l2', 'l1'], 'solver': ['liblinear', 'saga']},
            ]

